# Roteiro 3 — Ataques de inferência a modelos

Cibersegurança Aplicada com IA · Insper 2026-2

Cenário: um banco expõe por API um modelo de risco de login (RandomForest). Temos apenas acesso de consulta (`predict` / `predict_proba`) e alguns registros. O objetivo é medir o que o modelo vaza sobre seus dados de treino.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib
import torch
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

from art.estimators.classification.scikitlearn import ScikitlearnRandomForestClassifier
from art.attacks.inference.membership_inference import (
    MembershipInferenceBlackBox, MembershipInferenceBlackBoxRuleBased)
from art.attacks.inference.attribute_inference import AttributeInferenceBlackBox

DADOS = 'dados/'

## a) Preparação e diagnóstico

In [2]:
alvo = joblib.load(DADOS + 'alvo.joblib')
membros = pd.read_csv(DADOS + 'membros.csv.gz')
nao_membros = pd.read_csv(DADOS + 'nao_membros.csv.gz')
pool = pd.read_csv(DADOS + 'pool_consultas.csv.gz')

FEATURES = [c for c in membros.columns if c != 'fraude']
X_mem, y_mem = membros[FEATURES].values, membros['fraude'].values
X_nao, y_nao = nao_membros[FEATURES].values, nao_membros['fraude'].values

print(alvo)
print('features do alvo:', alvo.n_features_in_, '->', FEATURES)
print('membros', membros.shape, '| nao_membros', nao_membros.shape, '| pool', pool.shape)
print(f'taxa de fraude: membros={y_mem.mean():.3f}  nao_membros={y_nao.mean():.3f}')

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=20260401)
features do alvo: 11 -> ['hora', 'geo_dist_km', 'disp_idade_dias', 'num_tentativas_1h', 'vel_digitacao', 'rep_asn', 'vpn', 'novo_pais', 'score_comportamento', 'valor_sessao', 'conta_premium']
membros (4000, 12) | nao_membros (4000, 12) | pool (8000, 11)
taxa de fraude: membros=0.132  nao_membros=0.151


In [3]:
pred_mem = alvo.predict(X_mem)
pred_nao = alvo.predict(X_nao)

acc_treino = accuracy_score(y_mem, pred_mem)
acc_teste = accuracy_score(y_nao, pred_nao)
f1_teste = f1_score(y_nao, pred_nao)
gap = acc_treino - acc_teste
baseline = 1 - y_nao.mean()  # chutar sempre 'legítima'

print(f'Acurácia de treino (membros):     {acc_treino:.4f}')
print(f'Acurácia de teste (não-membros):  {acc_teste:.4f}')
print(f'F1 de teste (classe fraude):      {f1_teste:.4f}')
print(f'Gap treino - teste:               {gap:.4f}')
print(f'Baseline (sempre legítima):       {baseline:.4f}')
print('\nMatriz de confusão (teste):')
print(confusion_matrix(y_nao, pred_nao))

Acurácia de treino (membros):     1.0000
Acurácia de teste (não-membros):  0.8650
F1 de teste (classe fraude):      0.3284
Gap treino - teste:               0.1350
Baseline (sempre legítima):       0.8485

Matriz de confusão (teste):
[[3328   66]
 [ 474  132]]


In [4]:
# Confiança do alvo na classe verdadeira: membros vs não-membros
conf_mem = alvo.predict_proba(X_mem)[np.arange(len(y_mem)), y_mem]
conf_nao = alvo.predict_proba(X_nao)[np.arange(len(y_nao)), y_nao]
print(pd.DataFrame({'membros': pd.Series(conf_mem).describe(),
                    'nao_membros': pd.Series(conf_nao).describe()}).round(3))

        membros  nao_membros
count  4000.000     4000.000
mean      0.934        0.806
std       0.087        0.243
min       0.575        0.000
25%       0.910        0.725
50%       0.970        0.915
75%       0.995        0.980
max       1.000        1.000


**Comentário (a)**

- **Treino 100,0% vs teste 86,5% → gap de 13,5 p.p.** O RandomForest sem limite de profundidade decorou todos os membros.
- **A acurácia de teste é enganosa:** só ~15% das sessões são fraude, então um modelo que responde sempre "legítima" já acerta 84,9%. Os 86,5% ficam apenas 1,6 p.p. acima desse chute. O F1 da classe fraude (0,33) e a matriz de confusão mostram o problema real: o alvo deixa passar 474 das 606 fraudes (recall ≈ 22%). Em problema desbalanceado a acurácia é dominada pela classe majoritária e esconde o desempenho fraco na classe que importa.
- **O gap é a condição do membership inference:** o modelo se comporta de forma diferente em pontos que viu e em pontos que não viu. Nos membros ele acerta sempre e com confiança alta (mediana 0,97, mínimo 0,575); nos não-membros a confiança na classe correta é menor e mais dispersa (média 0,81, com casos em 0). Essa diferença de distribuição é o sinal que o atacante explora: saída "correta e muito confiante" → provável membro. Sem gap (modelo que generaliza), as duas distribuições coincidiriam e o ataque não teria em que se apoiar.

## b) Membership inference por regra

In [5]:
# Alvo embrulhado na ART: só usamos predict/predict_proba (acesso de API)
art_alvo = ScikitlearnRandomForestClassifier(model=alvo)

# Divisão pedida: primeiras 2000 linhas = treino do ataque, 2000 restantes = avaliação
N = 2000
X_mem_tr, y_mem_tr, X_mem_ev, y_mem_ev = X_mem[:N], y_mem[:N], X_mem[N:], y_mem[N:]
X_nao_tr, y_nao_tr, X_nao_ev, y_nao_ev = X_nao[:N], y_nao[:N], X_nao[N:], y_nao[N:]

def metricas_membership(inf_mem, inf_nao):
    """inf_* = predição do ataque (1 = membro) sobre membros e não-membros."""
    y_true = np.r_[np.ones(len(inf_mem)), np.zeros(len(inf_nao))]
    y_pred = np.r_[inf_mem, inf_nao]
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {'acuracia': accuracy_score(y_true, y_pred),
            'precisao': precision_score(y_true, y_pred),
            'recall_TPR': tp / (tp + fn),
            'FPR': fp / (fp + tn)}

In [6]:
ataque_regra = MembershipInferenceBlackBoxRuleBased(art_alvo)

# Regra: 'membro' se o alvo classifica o registro corretamente
inf_mem_regra = ataque_regra.infer(X_mem_ev, y_mem_ev).ravel()
inf_nao_regra = ataque_regra.infer(X_nao_ev, y_nao_ev).ravel()

res_regra = metricas_membership(inf_mem_regra, inf_nao_regra)
for k, v in res_regra.items():
    print(f'{k:12s} {v:.4f}')

acuracia     0.5693
precisao     0.5372
recall_TPR   1.0000
FPR          0.8615


**Comentário (b)**

| Métrica | Valor |
|---|---|
| Acurácia do ataque | 0,569 |
| Precisão | 0,537 |
| TPR (recall) | 1,000 |
| FPR | 0,862 |

A regra é simples: o ataque diz "membro" sempre que o alvo acerta o rótulo. Como o alvo decorou o treino (acurácia 100% nos membros), **todos os membros são detectados (TPR = 1,0)**. O problema está nos não-membros. O alvo também acerta cerca de 86% deles, porque a maioria é sessão legítima, que é a classe majoritária e fácil de acertar. Cada acerto vira um falso "membro", e por isso a **FPR é 0,86**, praticamente igual à acurácia de teste do alvo.

Em outras palavras, a FPR alta não vem de uma falha do ataque. Ela é consequência direta da acurácia enganosa discutida no item a): neste problema desbalanceado, "acertar" não distingue membro de não-membro. O único sinal que a regra aproveita é o gap de 13,5 p.p. Os não-membros que o alvo erra (~14%) são os únicos corretamente classificados como "não-membro". Por isso a acurácia do ataque fica em ~0,57, pouco acima do acaso (0,5), e a precisão em 0,54: quando o ataque diz "membro", ele acerta pouco mais que uma moeda. A regra usa só a saída binária (acertou/errou) e ignora o valor da confiança. O ataque treinado do item c) usa justamente a confiança.

## c) Membership inference treinado (black-box)

O ataque recebe o vetor de confianças do alvo (`predict_proba`) junto com o rótulo verdadeiro e aprende a separar membro de não-membro. Usamos o modelo de ataque padrão da ART (`attack_model_type='nn'`, rede neural em PyTorch). Além de `np.random.seed(42)`, fixamos `torch.manual_seed(42)`: a rede inicializa os pesos pelo gerador do PyTorch, e sem essa semente o resultado varia entre execuções.

In [7]:
def treinar_membership(estimador, tipo='nn'):
    np.random.seed(42)
    torch.manual_seed(42)
    ataque = MembershipInferenceBlackBox(estimador, attack_model_type=tipo, input_type='prediction')
    ataque.fit(X_mem_tr, y_mem_tr, X_nao_tr, y_nao_tr)
    inf_mem = ataque.infer(X_mem_ev, y_mem_ev).ravel()
    inf_nao = ataque.infer(X_nao_ev, y_nao_ev).ravel()
    return metricas_membership(inf_mem, inf_nao)

res_treinado = treinar_membership(art_alvo, 'nn')
for k, v in res_treinado.items():
    print(f'{k:12s} {v:.4f}')

acuracia     0.6717
precisao     0.6318
recall_TPR   0.8235
FPR          0.4800


In [8]:
comparacao = pd.DataFrame({
    'acaso': {'acuracia': 0.5, 'precisao': 0.5, 'recall_TPR': np.nan, 'FPR': np.nan},
    'b) regra': res_regra,
    'c) treinado (nn)': res_treinado,
}).T
print(comparacao.round(4))

# Robustez: outros modelos de ataque da ART
print()
print(pd.DataFrame({t: treinar_membership(art_alvo, t) for t in ['rf', 'gb', 'lr', 'knn']}).T.round(4))

                  acuracia  precisao  recall_TPR     FPR
acaso               0.5000    0.5000         NaN     NaN
b) regra            0.5692    0.5372      1.0000  0.8615
c) treinado (nn)    0.6718    0.6318      0.8235  0.4800



     acuracia  precisao  recall_TPR     FPR
rf     0.6662    0.6259      0.8265  0.4940
gb     0.6725    0.6257      0.8585  0.5135
lr     0.5762    0.5539      0.7840  0.6315
knn    0.6480    0.5892      0.9780  0.6820


**Comentário (c)**

| Ataque | Acurácia | Precisão | Recall (TPR) | FPR |
|---|---|---|---|---|
| Acaso | 0,500 | 0,500 | – | – |
| b) Regra | 0,569 | 0,537 | 1,000 | 0,862 |
| c) Treinado (nn, seed 42) | **0,672** | **0,632** | 0,824 | **0,480** |

O ataque treinado **supera a regra em ~10 p.p. de acurácia e fica ~17 p.p. acima do acaso**. O ganho vem da confiança. A regra só via "acertou/errou", que vale para 86% dos não-membros. O ataque treinado enxerga *quão confiante* foi o acerto e aprende que membros recebem probabilidades quase extremas (mediana 0,97 na classe certa), enquanto não-membros acertados costumam receber valores intermediários. Ele abre mão de parte do recall (1,00 → 0,82) para cortar a FPR quase pela metade (0,86 → 0,48), e a precisão sobe de 0,54 para 0,63. O resultado se mantém com outros classificadores de ataque (rf 0,666; gb 0,673; knn 0,648), o que mostra que o vazamento está na saída do alvo e não depende do modelo de ataque escolhido. Só a regressão logística (0,576) rende menos, porque a fronteira entre as confianças de membros e não-membros não é linear.

Aqui o atacante recebeu exemplos reais rotulados de membro/não-membro. Na prática esses rótulos vêm de **modelos-sombra (AML.T0005.001)**: o atacante treina cópias do alvo com dados de distribuição parecida, sabe exatamente quem entrou no treino de cada sombra e usa as saídas delas para treinar o classificador de ataque, que depois é aplicado ao alvo real.

**Privacidade e LGPD.** Saber que uma pessoa está no conjunto de treino já revela um dado sobre ela, mesmo sem acesso a nenhum atributo. Aqui, pertencer ao treino significa ser cliente do banco e ter tido uma sessão de login registrada e possivelmente rotulada como fraude. Em outros contextos (um modelo treinado só com pacientes de uma doença, ou só com vítimas de golpe), a pertença revela por si só uma condição sensível. A LGPD (Lei 13.709/2018) protege qualquer informação relacionada a pessoa natural identificada ou identificável (art. 5º, I), exige que o tratamento respeite finalidade, necessidade e segurança (art. 6º, I, III e VII) e trata como dado sensível, com proteção reforçada, informações como saúde e origem racial (arts. 5º, II, e 11). O modelo também é uma forma de tratamento desses dados. Se a API permite inferir quem foi usado no treino, o controlador não está adotando as medidas de segurança exigidas pelo art. 46, e uma exploração bem-sucedida pode configurar incidente de segurança que precisa ser comunicado à ANPD e aos titulares (art. 48). O modelo publicado vira um canal de vazamento dos dados que deveria proteger.

## d) Attribute inference: recuperar `conta_premium`

O ataque é treinado sobre os membros: para cada registro, a ART consulta o alvo e aprende a mapear (demais features + classe prevista pelo alvo) → `conta_premium`. Depois, nos não-membros, recebe **apenas** as 10 outras features e a predição do alvo. A coluna sensível nunca é entregue ao ataque.

In [9]:
IDX_PREMIUM = FEATURES.index('conta_premium')
OUTRAS = [i for i in range(len(FEATURES)) if i != IDX_PREMIUM]
premium_nao = X_nao[:, IDX_PREMIUM].astype(int)

np.random.seed(42)
torch.manual_seed(42)
ataque_attr = AttributeInferenceBlackBox(art_alvo, attack_feature=IDX_PREMIUM)  # attack_model_type='nn' (padrão)
ataque_attr.fit(X_mem)

# Na inferência: só as outras features + a resposta da API (classe prevista)
pred_api_nao = alvo.predict(X_nao).reshape(-1, 1)
premium_inferido = ataque_attr.infer(X_nao[:, OUTRAS], pred=pred_api_nao, values=[0, 1]).ravel().astype(int)

classe_maj = int(np.bincount(premium_nao).argmax())
baseline_attr = (premium_nao == classe_maj).mean()
print(f'Proporção premium (não-membros): {premium_nao.mean():.4f}')
print(f'Baseline (sempre {classe_maj}):            {baseline_attr:.4f}')
print(f'Acurácia do ataque:              {accuracy_score(premium_nao, premium_inferido):.4f}')
print(f'Precisão (premium=1):            {precision_score(premium_nao, premium_inferido):.4f}')
print(f'Recall (premium=1):              {recall_score(premium_nao, premium_inferido):.4f}')
print('\nMatriz de confusão (linhas = real, colunas = inferido):')
print(confusion_matrix(premium_nao, premium_inferido))

Proporção premium (não-membros): 0.2955
Baseline (sempre 0):            0.7045
Acurácia do ataque:              0.7903
Precisão (premium=1):            0.6589
Recall (premium=1):              0.6015

Matriz de confusão (linhas = real, colunas = inferido):
[[2450  368]
 [ 471  711]]


In [10]:
# De onde vem o sinal? Relação entre conta_premium, fraude e a saída do alvo
df_nao = nao_membros.assign(pred_alvo=pred_api_nao.ravel())
print('Taxa de premium por classe prevista pelo alvo (não-membros):')
print(df_nao.groupby('pred_alvo')['conta_premium'].agg(['mean', 'size']).round(3))
print('\nTaxa de fraude real por conta_premium:')
print(nao_membros.groupby('conta_premium')['fraude'].mean().round(3))
print('\nImportância de conta_premium no alvo:',
      round(dict(zip(FEATURES, alvo.feature_importances_))['conta_premium'], 4))

# Ablação: o mesmo ataque, mas sem a predição do alvo (pred constante) -> só as outras features
pred_cte = np.zeros_like(pred_api_nao)
sem_pred = ataque_attr.infer(X_nao[:, OUTRAS], pred=pred_cte, values=[0, 1]).ravel().astype(int)
print(f'\nAcurácia com pred do alvo: {accuracy_score(premium_nao, premium_inferido):.4f}'
      f'  |  com pred zerada: {accuracy_score(premium_nao, sem_pred):.4f}')

Taxa de premium por classe prevista pelo alvo (não-membros):
            mean  size
pred_alvo             
0          0.297  3802
1          0.258   198

Taxa de fraude real por conta_premium:
conta_premium
0.0    0.147
1.0    0.163
Name: fraude, dtype: float64

Importância de conta_premium no alvo: 0.014

Acurácia com pred do alvo: 0.7903  |  com pred zerada: 0.7897


In [11]:
# Quanto do acerto vem só dos dados? Correlação de conta_premium com as outras features
print('Correlação com conta_premium (não-membros):')
print(nao_membros[FEATURES].corr()['conta_premium'].drop('conta_premium').round(3).sort_values().tail(3))

# Baseline "só dados": mesmo tipo de inferência, mas SEM consultar o alvo
so_dados = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=7)
so_dados.fit(X_mem[:, OUTRAS], X_mem[:, IDX_PREMIUM].astype(int))
acc_so_dados = accuracy_score(premium_nao, so_dados.predict(X_nao[:, OUTRAS]))
print(f'\nAcurácia só com as outras features (sem o alvo): {acc_so_dados:.4f}')

# Inversão pela confiança: a API devolve predict_proba para o registro real da vítima.
# O atacante testa os dois valores possíveis de conta_premium e fica com o que reproduz essa saída.
proba_obs = alvo.predict_proba(X_nao)[:, 1]          # o que a API respondeu para a vítima
X0, X1 = X_nao.copy(), X_nao.copy()
X0[:, IDX_PREMIUM], X1[:, IDX_PREMIUM] = 0, 1
d0 = np.abs(alvo.predict_proba(X0)[:, 1] - proba_obs)
d1 = np.abs(alvo.predict_proba(X1)[:, 1] - proba_obs)
empate = d0 == d1                                     # alvo indiferente ao atributo nesse ponto
inv = np.where(empate, classe_maj, (d1 < d0).astype(int))
print(f'Acurácia da inversão pela confiança:              {accuracy_score(premium_nao, inv):.4f}'
      f'  (empates resolvidos pela majoritária: {empate.mean():.1%})')
print(f'Acurácia fora dos empates:                        {accuracy_score(premium_nao[~empate], inv[~empate]):.4f}')
print(f'Precisão / recall premium:                        {precision_score(premium_nao, inv):.4f} / {recall_score(premium_nao, inv):.4f}')

Correlação com conta_premium (não-membros):
rep_asn                0.021
score_comportamento    0.335
valor_sessao           0.370
Name: conta_premium, dtype: float64



Acurácia só com as outras features (sem o alvo): 0.7975
Acurácia da inversão pela confiança:              0.9245  (empates resolvidos pela majoritária: 25.1%)
Acurácia fora dos empates:                        1.0000
Precisão / recall premium:                        1.0000 / 0.7445


**Comentário (d)**

| Abordagem | O que o atacante usa | Acurácia | Precisão (premium) | Recall (premium) |
|---|---|---|---|---|
| Baseline (sempre "não premium") | nada | 0,705 | – | 0,000 |
| **`AttributeInferenceBlackBox` (ART)** | outras features + classe prevista | **0,790** | **0,659** | **0,602** |
| Só dados (sem consultar o alvo) | outras features | 0,798 | – | – |
| Inversão pela confiança | outras features + `predict_proba` | 0,925 | 1,000 | 0,745 |

O ataque da ART recupera `conta_premium` de não-membros com **79,0% de acurácia, 8,6 p.p. acima do baseline**: 711 dos 1182 clientes premium são identificados, e 66% dos apontados como premium realmente são. A coluna sensível nunca foi entregue ao ataque.

A investigação mostra **de onde vem esse vazamento**:

1. **Correlação nos dados.** `conta_premium` está correlacionada com `valor_sessao` (0,37) e `score_comportamento` (0,34): clientes premium movimentam mais e têm comportamento mais estável. Só com essas features, sem consultar o alvo, chega-se a ~0,80, e a ablação do item (pred zerada → 0,790) confirma que, neste dataset, a classe prevista (0/1) quase não acrescenta informação. Grande parte do ganho é "inferência estatística" a partir dos dados públicos.
2. **A saída do modelo como canal.** O alvo usa `conta_premium` como entrada, então sua resposta é uma função desse atributo. Quanto mais granular a saída, mais ela carrega. Um rótulo binário, com 95% das respostas iguais a "legítima", tem quase nenhuma informação. Já a **probabilidade** é quase uma impressão digital do registro: consultando a API com `conta_premium = 0` e `= 1` e escolhendo o valor que reproduz a confiança observada da vítima, o atacante recupera o atributo com **100% de acerto em todos os casos em que o modelo depende dele** (75% dos registros). Nos outros 25%, nenhuma árvore usa a coluna naquele ponto, e o melhor é chutar a majoritária. Resultado: **92,5% de acurácia e precisão 1,0**. É a inversão de modelo clássica (Fredrikson et al.), e ela usa apenas a saída da API.

Em outras palavras: ao aprender a relação entre `conta_premium` e as demais variáveis para prever fraude, o modelo passa a codificar essa coluna na própria resposta. Quem conhece as outras features de uma pessoa (ou as obtém de outra fonte) e vê a saída do modelo para ela consegue "desfazer" a função e recuperar o atributo sensível. Isso antecipa a defesa do item f): reduzir a granularidade da saída (devolver só o rótulo, ou probabilidades arredondadas) fecha a maior parte desse canal.

## e) Extração e model stealing (AML.T0024.002)

O atacante não tem os rótulos verdadeiros do pool: ele envia cada linha à API e guarda a resposta do alvo. Esses pares (consulta, resposta) são o conjunto de treino do clone.

In [12]:
# 1) Consultas à "API": o único acesso é a saída do alvo para as linhas do pool
X_pool = pool[FEATURES].values
y_pool_api = alvo.predict(X_pool)
print(f'Consultas feitas: {len(X_pool)}  |  rotuladas como fraude pela API: {y_pool_api.mean():.3f}')

# 2) Surrogate treinado só nos pares (consulta, resposta da API)
surrogate = RandomForestClassifier(random_state=7)
surrogate.fit(X_pool, y_pool_api)

# 3) Avaliação nos não-membros
pred_sur_nao = surrogate.predict(X_nao)
fidelidade = (pred_sur_nao == pred_nao).mean()        # concorda com o alvo?
acc_sur = accuracy_score(y_nao, pred_sur_nao)         # acerta o rótulo real?
print(f'Fidelidade (surrogate == alvo, não-membros): {fidelidade:.4f}')
print(f'Acurácia real do surrogate:                  {acc_sur:.4f}   (alvo: {acc_teste:.4f})')
print(f'F1 do surrogate:                             {f1_score(y_nao, pred_sur_nao):.4f}   (alvo: {f1_teste:.4f})')
print('\nConcordância por classe prevista pelo alvo:')
print(pd.crosstab(pd.Series(pred_nao, name='alvo'), pd.Series(pred_sur_nao, name='surrogate')))

Consultas feitas: 8000  |  rotuladas como fraude pela API: 0.046


Fidelidade (surrogate == alvo, não-membros): 0.9750
Acurácia real do surrogate:                  0.8610   (alvo: 0.8650)
F1 do surrogate:                             0.2213   (alvo: 0.3284)

Concordância por classe prevista pelo alvo:
surrogate     0    1
alvo                
0          3797    5
1            95  103


In [13]:
# Quantas consultas são necessárias? Fidelidade em função do orçamento de consultas
orcamentos = [100, 250, 500, 1000, 2000, 4000, 8000]
curva = []
for n in orcamentos:
    s = RandomForestClassifier(random_state=7).fit(X_pool[:n], y_pool_api[:n])
    p = s.predict(X_nao)
    curva.append({'consultas': n, 'fidelidade': (p == pred_nao).mean(), 'acuracia': accuracy_score(y_nao, p)})
print(pd.DataFrame(curva).round(4).to_string(index=False))

 consultas  fidelidade  acuracia
       100      0.9505    0.8485
       250      0.9512    0.8498
       500      0.9532    0.8512
      1000      0.9602    0.8548
      2000      0.9648    0.8588
      4000      0.9672    0.8592
      8000      0.9750    0.8610


In [14]:
# Evasão offline: o atacante usa o clone para ajustar uma fraude até ela "passar",
# sem consultar a API. Só no fim testa no alvo real (1 consulta por tentativa).
# Mexe apenas em sinais que o fraudador controla: VPN, ASN, horário, ritmo de digitação, nº de tentativas.
mascara = (y_nao == 1) & (pred_nao == 1)                       # fraudes que o alvo pega hoje
fraudes = X_nao[mascara]
idx = {f: FEATURES.index(f) for f in FEATURES}
legit = membros[membros.fraude == 0]                            # perfil típico de sessão legítima
alvos_legit = {'vpn': 0, 'rep_asn': 1, 'novo_pais': 0,
               'num_tentativas_1h': legit.num_tentativas_1h.median(),
               'hora': legit.hora.median(),
               'vel_digitacao': legit.vel_digitacao.median()}

def evadir_offline(x, modelo):
    """Aplica, uma a uma, as mudanças até o modelo dizer 'legítima'. Devolve (x', nº de mudanças)."""
    x = x.copy()
    for k, (f, v) in enumerate(alvos_legit.items(), start=1):
        x[idx[f]] = v
        if modelo.predict(x.reshape(1, -1))[0] == 0:
            return x, k
    return x, None

resultados = [evadir_offline(x, surrogate) for x in fraudes]
adv = np.array([r[0] for r in resultados])
passou_clone = np.array([r[1] is not None for r in resultados])
passou_alvo = alvo.predict(adv) == 0

print(f'Fraudes detectadas pelo alvo:                 {len(fraudes)}')
print(f'Evadem o clone (trabalho 100% offline):       {passou_clone.mean():.1%}')
print(f'Dessas, também evadem o alvo real (transfer): {passou_alvo[passou_clone].mean():.1%}')
print(f'Taxa de evasão no alvo, no total:             {passou_alvo.mean():.1%}'
      f'  ({len(fraudes)} consultas à API, uma por fraude)')

Fraudes detectadas pelo alvo:                 132
Evadem o clone (trabalho 100% offline):       100.0%
Dessas, também evadem o alvo real (transfer): 59.8%
Taxa de evasão no alvo, no total:             59.8%  (132 consultas à API, uma por fraude)


**Comentário (e)**

| Métrica (não-membros) | Surrogate | Alvo |
|---|---|---|
| Fidelidade (concorda com o alvo) | **0,975** | – |
| Acurácia real | **0,861** | 0,865 |
| F1 (fraude) | 0,221 | 0,328 |

Com **8000 consultas** e nenhum rótulo verdadeiro, o clone concorda com o alvo em 97,5% dos não-membros e fica só 0,4 p.p. abaixo dele em acurácia real. O atacante obteve, de graça, um modelo praticamente equivalente ao que o banco treinou com dados rotulados de fraude.

**Leitura crítica da fidelidade.** Como o alvo responde "legítima" em ~95% dos casos, um clone que sempre diz 0 já teria fidelidade 0,9505. É exatamente o valor da curva com só 100 consultas. O ganho real está na classe rara: o clone reproduz 103 das 198 detecções de fraude do alvo (52%), e quase não inventa fraude onde o alvo não vê (5 casos). A fidelidade cresce de forma contínua com o orçamento (0,950 → 0,960 → 0,975 entre 100, 1000 e 8000 consultas). Com mais consultas, sobretudo mais próximas da fronteira de decisão, o clone tende a se aproximar ainda mais. Por isso volume e padrão de consultas são o principal sinal de detecção desse ataque.

**Consequência 1: roubo de propriedade intelectual.** O valor do modelo está nos dados rotulados de fraude, no feature engineering e no ajuste, que custaram tempo e dinheiro ao banco. O atacante replicou esse valor pagando apenas o custo de consultas a uma API, sem acesso a pesos, código ou dados. O clone pode ser revendido, usado por um concorrente ou servir para reduzir a vantagem competitiva do banco. E ele herda também os vieses e o que o modelo memorizou, o que reabre os ataques de privacidade dos itens b) a d) sem limite de consultas.

**Consequência 2: evasão offline.** Com o clone em mãos, o atacante ensaia ataques de evasão (AML.T0015) sem tocar na API e, portanto, sem disparar rate limit ou alertas. No experimento acima, o fraudador parte de 132 fraudes que o alvo **detecta hoje** e ajusta, contra o clone, apenas sinais que ele controla (desligar VPN, usar um ASN de boa reputação, logar no horário e com o ritmo de digitação típicos, poucas tentativas). As 132 passam no clone, e **59,8% delas também passam no alvo real**, com uma única consulta por fraude, já na forma final. A transferência não é perfeita (o clone não é o alvo), mas o atacante converte a maior parte das detecções em evasões com custo de consulta mínimo e sem exploração visível.

**Mapeamento: MITRE ATLAS AML.T0024.002, Extract AI Model** (sub-técnica de *Exfiltration via AI Inference API*). O modelo é exfiltrado só por consultas à API de inferência. No NIST AI 100-2e2025, pertence à família de **ataques de privacidade**, na categoria **model extraction**.